# ETL 실습 — 흩어진 데이터를 모아 정제하고 적재하기

실무 데이터는 한 곳에 있지 않습니다. 이 실습에서는 세 원천에서 데이터를 **추출(Extract)** 하고,
전처리 4대 항목으로 **변환(Transform)** 한 뒤, 데이터 레이크에 **적재(Load)** 합니다.

| 원천 | 시스템 | 접근 방식 |
|---|---|---|
| 회원/주문 | MySQL (`shop-mysql`) | JDBC |
| 채널 매출 | REST API (`sales-api`) | requests |
| 웹 로그 | MinIO 데이터 레이크 | s3a + Parquet |

In [ ]:
# 0) SparkSession 생성 — JDBC 드라이버와 S3 커넥터를 함께 로드
#    (최초 실행 시 라이브러리를 내려받느라 1~2분 걸릴 수 있습니다)
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("etl-lab")
    .config("spark.jars.packages",
            "com.mysql:mysql-connector-j:8.3.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4")
    # ── s3a → MinIO 연결 설정 ──
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "minio12345")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)
print("SparkSession OK — version", spark.version)

In [ ]:
# 0-1) [준비 셀 — 1회만] 원천 시스템이 데이터 레이크에 로그를 적재해 둔 상황 재현
logs = spark.createDataFrame(
    [(1, "/home", "2026-04-29 09:01"), (2, "/product/7", "2026-04-29 09:02"),
     (3, "/cart", "2026-04-29 09:05"), (1, "/order", "2026-04-29 09:07"),
     (5, "/home", "2026-04-29 09:10")],
    ["member_id", "page", "ts"],
)
logs.write.mode("overwrite").parquet("s3a://my-datalake/logs/")
print("레이크에 로그 적재 완료 → http://localhost:9001 (admin/minio12345)에서 확인 가능")

## 1. Extract — 세 원천에서 데이터 가져오기

In [ ]:
# 1-1) DB 연동 (JDBC) — 병렬 로딩 옵션 포함
jdbc_url = "jdbc:mysql://mysql:3306/shop"
db_props = {"user": "analyst", "password": "analyst123",
            "driver": "com.mysql.cj.jdbc.Driver"}

# partitionColumn/lowerBound/upperBound/numPartitions은 스파크에게
# "member_id를 4개 구간으로 나눠서 4개 커넥션으로 동시에 읽어라"고 알려주는 옵션입니다.
# 이 옵션이 없으면 스파크가 테이블 전체를 커넥션 1개로만 읽어서, 파티션 수가 1이 되고
# 대용량 테이블에서는 그만큼 느립니다. lowerBound/upperBound는 실제 min/max와
# 정확히 같을 필요는 없고, 대략적인 범위만 줘도 스파크가 구간을 나눠줍니다.
df_members = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "members")
    .option("user", "analyst").option("password", "analyst123")
    .option("partitionColumn", "member_id")
    .option("lowerBound", "1").option("upperBound", "11")
    .option("numPartitions", "4")
    .load()
)
df_orders = spark.read.jdbc(jdbc_url, "orders", properties=db_props)

df_members.show()
df_orders.show()
df_members.printSchema()  # ⚠️ age와 amount가 string인 점에 주목!
# (MySQL에서 VARCHAR로 저장해뒀기 때문입니다 — 이후 2-1 단계에서 타입 변환으로 고칩니다)

In [ ]:
# 1-2) REST API 연동
import requests

response = requests.get("http://sales-api:5000/sales",
                        params={"date": "2026-04-29"})
df_api = spark.createDataFrame(response.json()["records"])
df_api.show()

In [ ]:
# 1-3) Data Lake 연동 (s3a + Parquet)
df_logs = spark.read.parquet("s3a://my-datalake/logs/")
df_logs.show()

## 2. Transform — 전처리에서 반드시 챙겨야 할 4가지

위에서 추출한 데이터에는 문제가 심어져 있습니다. 하나씩 찾아 정제합니다.
1. **결측치** — 이메일이 NULL인 회원
2. **이상치** — 나이 250세, 음수 주문, 100억 주문
3. **중복** — 두 번 적재된 회원·주문
4. **타입** — 문자열로 들어온 age, amount

In [ ]:
# 2-1) 타입 변환 — "문자열 '123'은 숫자가 아니다" (다른 정제보다 먼저!)
from pyspark.sql import functions as F

df_members = df_members.withColumn("age", F.col("age").cast("int"))
df_orders = df_orders.withColumn("amount", F.col("amount").cast("long"))
df_members.printSchema(); df_orders.printSchema()

# 💡 왜 먼저? 문자열 상태에선 사전식(lexicographic) 비교가 일어나
#    "01" < "2" 같은 엉뚱한 결과가 나옵니다. 예를 들어 amount가 아직 문자열이면
#    F.col("amount") > 0 같은 이상치 필터가 숫자 크기가 아니라 글자 순서로
#    비교되어 버려서, 다음 단계(2-3 이상치 처리)가 제대로 동작하지 않습니다.
#    그래서 타입 변환을 결측치·이상치 처리보다 앞에 둡니다.

In [ ]:
# 2-2) 결측치 처리 — "빈칸을 어떻게 할까"
print("이메일 결측 회원 수:", df_members.filter(F.col("email").isNull()).count())

df_members_clean = df_members.dropna(subset=["email"])   # 삭제 전략
# 또는 값 채우기 전략: df_members.fillna({"email": "unknown@example.com"})
print("처리 후 회원 수:", df_members_clean.count())

In [ ]:
# 2-3) 이상치 처리 — "비정상 값을 걸러내라"
# (a) 도메인 기준: 나이 0~120, 주문금액 > 0
df_members_clean = df_members_clean.filter(F.col("age").between(0, 120))
df_orders_clean = df_orders.filter(F.col("amount") > 0)

# (b) 통계 기준: IQR — Q1-1.5×IQR ~ Q3+1.5×IQR 밖 제거
# IQR(사분위 범위)은 "데이터의 가운데 50%가 퍼진 정도"입니다.
# 1.5배라는 계수는 통계학에서 관례적으로 쓰는 값으로, 정상 분포를 가정했을 때
# 이 범위를 벗어나는 값은 극단치로 볼 만하다는 경험적 기준입니다.
# approxQuantile의 세 번째 인자(0.0)는 오차 허용치로, 0이면 정확한 값을 계산합니다
# (대용량 데이터에서는 0보다 큰 값을 주면 근사치를 더 빠르게 계산합니다).
q1, q3 = df_orders_clean.approxQuantile("amount", [0.25, 0.75], 0.0)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"IQR 허용 범위: {low:,.0f} ~ {high:,.0f}")

df_orders_clean = df_orders_clean.filter(F.col("amount").between(low, high))
df_orders_clean.show()  # 100억 주문이 사라졌는지 확인

In [ ]:
# 2-4) 중복 제거 — "같은 데이터를 두 번 세지 마라"
print("중복 제거 전 주문 수:", df_orders_clean.count())
df_orders_clean = df_orders_clean.dropDuplicates()            # 완전 중복 제거
# 특정 키 기준: df_orders_clean.dropDuplicates(["order_id"])
df_members_clean = df_members_clean.dropDuplicates(["member_id"])
print("중복 제거 후 주문 수:", df_orders_clean.count())

In [ ]:
# 2-5) 통합 — 회원별 총 주문금액 요약 만들기
df_summary = (
    df_members_clean.join(df_orders_clean, "member_id")
    .groupBy("member_id", "name")
    .agg(F.sum("amount").alias("total_amount"))
    .orderBy("member_id")
)
df_summary.show()

## 3. Load — 데이터 레이크에 적재

In [ ]:
# 3-1) Parquet으로 적재
df_summary.write.mode("overwrite").parquet(
    "s3a://my-datalake/analytics/member_summary/")
print("적재 완료")

# 3-2) 다시 읽어서 검증 — ETL 파이프라인 한 바퀴 완성!
spark.read.parquet("s3a://my-datalake/analytics/member_summary/").show()

### 생각해 볼 질문
1. 이메일 결측 회원을 **삭제** 대신 **fillna**로 채운다면 이후 분석에 어떤 차이가 생길까요?
2. 타입 변환을 이상치 필터보다 **먼저** 해야 하는 이유는 무엇일까요?
3. 100억 주문이 입력 실수가 아니라 **진짜 거래**였다면, IQR로 지우는 게 맞을까요?
4. `dropDuplicates()`와 `dropDuplicates(["order_id"])`의 결과가 달라지는 상황은 언제일까요?

In [ ]:
spark.stop()